In [20]:
import requests
from bs4 import BeautifulSoup
import re


headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
}


def extract_performer_id(url):
    """
    Extract IBDB performer ID from URL.
    Example:
    /broadway-cast-staff/alfred-drake-4031 -> 4031
    """
    
    return url.split("-")[-1]


def extract_cast(url):
    """
    Extract Opening Night Cast from an IBDB production page.
    Returns a list of performers.
    """

    # Request page
    response = requests.get(
        url,
        headers=headers
    )

    response.raise_for_status()

    # Parse HTML
    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # Locate Opening Night Cast section
    cast_section = soup.find(
        id="OpeningNightCast"
    )

    if cast_section is None:
        raise ValueError(
            "Opening Night Cast section not found"
        )

    # Find individual cast rows
    cast_rows = cast_section.find_all(
        "div",
        class_="row mobile-a-align"
    )

    cast = []

    for row in cast_rows:

        performer = row.find(
            "a",
            href=re.compile(
                "/broadway-cast-staff/"
            )
        )

        # Ignore rows without performer links
        if performer is None:
            continue

        columns = row.find_all(
            "div",
            class_="col m4 s12"
        )

        character = None

        if len(columns) > 1:
            character = columns[1].get_text(
                strip=True
            )

        cast.append(
            {
                "performer_id": extract_performer_id(
                    performer["href"]
                ),
                "performer_name": performer.text.strip(),
                "character": character
            }
        )

    return cast

In [21]:
import pandas as pd

In [22]:
production = {
    "production_id": "1285",
    "production_title": "Oklahoma!",
    "opening_date": "Mar 31, 1943",
    "production_type": "Original"
}

url = "https://www.ibdb.com/broadway-production/oklahoma-1285"

In [23]:
cast = extract_cast(url)

cast[:5]

[{'performer_id': '4031',
  'performer_name': 'Alfred Drake',
  'character': 'Curly'},
 {'performer_id': '57977',
  'performer_name': 'Joan Roberts',
  'character': 'Laurey'},
 {'performer_id': '14313',
  'performer_name': 'Joseph Buloff',
  'character': 'Ali Hakim'},
 {'performer_id': '67218',
  'performer_name': 'Howard Da Silva',
  'character': 'Jud Fry'},
 {'performer_id': '38152',
  'performer_name': 'Lee Dixon',
  'character': 'Will Parker'}]

In [24]:
cast_df = pd.DataFrame(cast)

cast_df.head()

,performer_id,performer_name,character
0,4031,Alfred Drake,Curly
1,57977,Joan Roberts,Laurey
2,14313,Joseph Buloff,Ali Hakim
3,67218,Howard Da Silva,Jud Fry
4,38152,Lee Dixon,Will Parker


In [25]:
production_performer = cast_df.copy()

production_performer["production_id"] = production["production_id"]
production_performer["production_title"] = production["production_title"]
production_performer["opening_date"] = production["opening_date"]
production_performer["production_type"] = production["production_type"]

production_performer.head()

,performer_id,performer_name,character,production_id,production_title,opening_date,production_type
0,4031,Alfred Drake,Curly,1285,Oklahoma!,"Mar 31, 1943",Original
1,57977,Joan Roberts,Laurey,1285,Oklahoma!,"Mar 31, 1943",Original
2,14313,Joseph Buloff,Ali Hakim,1285,Oklahoma!,"Mar 31, 1943",Original
3,67218,Howard Da Silva,Jud Fry,1285,Oklahoma!,"Mar 31, 1943",Original
4,38152,Lee Dixon,Will Parker,1285,Oklahoma!,"Mar 31, 1943",Original


In [26]:
production_performer = production_performer[
    [
        "production_id",
        "production_title",
        "opening_date",
        "production_type",
        "performer_id",
        "performer_name",
        "character"
    ]
]

production_performer.head()

,production_id,production_title,opening_date,production_type,performer_id,performer_name,character
0,1285,Oklahoma!,"Mar 31, 1943",Original,4031,Alfred Drake,Curly
1,1285,Oklahoma!,"Mar 31, 1943",Original,57977,Joan Roberts,Laurey
2,1285,Oklahoma!,"Mar 31, 1943",Original,14313,Joseph Buloff,Ali Hakim
3,1285,Oklahoma!,"Mar 31, 1943",Original,67218,Howard Da Silva,Jud Fry
4,1285,Oklahoma!,"Mar 31, 1943",Original,38152,Lee Dixon,Will Parker


In [27]:
production_performer.shape

(35, 7)

In [28]:
production_performer["performer_id"].nunique()

35

In [29]:
duplicates = production_performer[
    production_performer.duplicated(
        subset=["production_id", "performer_id"],
        keep=False
    )
]

duplicates

,production_id,production_title,opening_date,production_type,performer_id,performer_name,character


In [30]:
production_performer_clean = (
    production_performer
    .groupby(
        [
            "production_id",
            "production_title",
            "opening_date",
            "production_type",
            "performer_id",
            "performer_name"
        ],
        as_index=False
    )
    .agg(
        {
            "character": lambda x: ", ".join(sorted(set(x)))
        }
    )
)

production_performer_clean.head()

,production_id,production_title,opening_date,production_type,performer_id,performer_name,character
0,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble
1,1285,Oklahoma!,"Mar 31, 1943",Original,101764,Carl Nelson,Singing Ensemble
2,1285,Oklahoma!,"Mar 31, 1943",Original,111506,Dorothea MacFarland,Singing Ensemble
3,1285,Oklahoma!,"Mar 31, 1943",Original,1197,June Graham,Laurey's FriendDream Ballet
4,1285,Oklahoma!,"Mar 31, 1943",Original,1219,Ray Harrison,Dancing Ensemble


In [31]:
edges = production_performer_clean.merge(
    production_performer_clean,
    on="production_id",
    suffixes=("_1", "_2")
)

edges.head()

,production_id,production_title_1,opening_date_1,production_type_1,performer_id_1,performer_name_1,character_1,production_title_2,opening_date_2,production_type_2,performer_id_2,performer_name_2,character_2
0,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble
1,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,101764,Carl Nelson,Singing Ensemble
2,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,111506,Dorothea MacFarland,Singing Ensemble
3,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,1197,June Graham,Laurey's FriendDream Ballet
4,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,1219,Ray Harrison,Dancing Ensemble


In [32]:
edges = edges[
    edges["performer_id_1"] != edges["performer_id_2"]
]

edges.head()

,production_id,production_title_1,opening_date_1,production_type_1,performer_id_1,performer_name_1,character_1,production_title_2,opening_date_2,production_type_2,performer_id_2,performer_name_2,character_2
1,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,101764,Carl Nelson,Singing Ensemble
2,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,111506,Dorothea MacFarland,Singing Ensemble
3,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,1197,June Graham,Laurey's FriendDream Ballet
4,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,1219,Ray Harrison,Dancing Ensemble
5,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,14313,Joseph Buloff,Ali Hakim


In [33]:
edges["pair"] = edges.apply(
    lambda row: tuple(
        sorted(
            [
                row["performer_id_1"],
                row["performer_id_2"]
            ]
        )
    ),
    axis=1
)

edges = edges.drop_duplicates("pair")

edges.head()

,production_id,production_title_1,opening_date_1,production_type_1,performer_id_1,performer_name_1,character_1,production_title_2,opening_date_2,production_type_2,performer_id_2,performer_name_2,character_2,pair
1,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,101764,Carl Nelson,Singing Ensemble,"(101763, 101764)"
2,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,111506,Dorothea MacFarland,Singing Ensemble,"(101763, 111506)"
3,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,1197,June Graham,Laurey's FriendDream Ballet,"(101763, 1197)"
4,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,1219,Ray Harrison,Dancing Ensemble,"(101763, 1219)"
5,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble,Oklahoma!,"Mar 31, 1943",Original,14313,Joseph Buloff,Ali Hakim,"(101763, 14313)"


In [36]:
performer_edges = edges[
    [
        "performer_id_1",
        "performer_name_1",
        "performer_id_2",
        "performer_name_2",
        "production_id",
        "production_title_1"
    ]
]

performer_edges.head()

performer_edges = performer_edges.rename(
    columns={
        "production_title_1": "production_title"
    }
)

performer_edges.head()

,performer_id_1,performer_name_1,performer_id_2,performer_name_2,production_id,production_title
1,101763,John Baum,101764,Carl Nelson,1285,Oklahoma!
2,101763,John Baum,111506,Dorothea MacFarland,1285,Oklahoma!
3,101763,John Baum,1197,June Graham,1285,Oklahoma!
4,101763,John Baum,1219,Ray Harrison,1285,Oklahoma!
5,101763,John Baum,14313,Joseph Buloff,1285,Oklahoma!


In [37]:
weighted_edges = (
    performer_edges
    .groupby(
        [
            "performer_id_1",
            "performer_name_1",
            "performer_id_2",
            "performer_name_2"
        ]
    )
    .size()
    .reset_index(name="shared_productions")
)

weighted_edges.head()

,performer_id_1,performer_name_1,performer_id_2,performer_name_2,shared_productions
0,101763,John Baum,101764,Carl Nelson,1
1,101763,John Baum,111506,Dorothea MacFarland,1
2,101763,John Baum,1197,June Graham,1
3,101763,John Baum,1219,Ray Harrison,1
4,101763,John Baum,14313,Joseph Buloff,1


In [38]:
print("Production-performer table:")
display(production_performer_clean.head())

print("\nEdge table:")
display(weighted_edges.head())

print("\nNumber of performers:")
print(
    production_performer_clean["performer_id"].nunique()
)

print("\nNumber of edges:")
print(
    len(weighted_edges)
)

Production-performer table:


,production_id,production_title,opening_date,production_type,performer_id,performer_name,character
0,1285,Oklahoma!,"Mar 31, 1943",Original,101763,John Baum,Singing Ensemble
1,1285,Oklahoma!,"Mar 31, 1943",Original,101764,Carl Nelson,Singing Ensemble
2,1285,Oklahoma!,"Mar 31, 1943",Original,111506,Dorothea MacFarland,Singing Ensemble
3,1285,Oklahoma!,"Mar 31, 1943",Original,1197,June Graham,Laurey's FriendDream Ballet
4,1285,Oklahoma!,"Mar 31, 1943",Original,1219,Ray Harrison,Dancing Ensemble



Edge table:


,performer_id_1,performer_name_1,performer_id_2,performer_name_2,shared_productions
0,101763,John Baum,101764,Carl Nelson,1
1,101763,John Baum,111506,Dorothea MacFarland,1
2,101763,John Baum,1197,June Graham,1
3,101763,John Baum,1219,Ray Harrison,1
4,101763,John Baum,14313,Joseph Buloff,1



Number of performers:
35

Number of edges:
595


In [39]:
print(
    "Average connections per performer:"
)

degree = (
    weighted_edges["performer_id_1"]
    .value_counts()
    +
    weighted_edges["performer_id_2"]
    .value_counts()
)

degree.describe()

Average connections per performer:


count    33.0
mean     34.0
std       0.0
min      34.0
25%      34.0
50%      34.0
75%      34.0
max      34.0
Name: count, dtype: float64